# Historical project notebook — not validated for deployment

This notebook preserves the team’s original workflow for transparency. **It contains known validation problems**: SMOTE before train/validation splitting, preprocessing fitted outside cross-validation, combined categorical encoding, and separate fitting on the unlabeled holdout. Its published performance figures must not be treated as unbiased prospective estimates.

The original dataset is intentionally **not** included. Place an authorized copy at `data/Insurance Data.xlsx` to examine the workflow; results may not reproduce without the original environment. For methodological improvements, see `risk_profiling_revised.ipynb`.


## Data Preparation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import re
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel('data/Insurance Data.xlsx')

## Data Understanding

In [ ]:
df.head()

In [ ]:
print("Dimensi data:")
print(df.shape)

print("\nTipe data setiap kolom:")
print(df.dtypes)

## Splitting Data

In [ ]:
def split_by_risk_profile(df: pd.DataFrame) -> (pd.DataFrame, pd.DataFrame):
    """
    Memisahkan DataFrame menjadi set training dan test berdasarkan
    keberadaan nilai di kolom 'Risk Profile'.

    Args:
        df (pd.DataFrame): DataFrame input yang sudah dimuat.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: Tuple berisi (df_train, df_test).
                                           Mengembalikan (None, None) jika kolom
                                           'Risk Profile' tidak ditemukan.
    """
    target_column = 'Risk Profile'

    # Pastikan kolom 'Risk Profile' ada dalam DataFrame
    if target_column not in df.columns:
        print(f"Error: Kolom '{target_column}' tidak ditemukan dalam DataFrame.")
        print(f"Kolom yang tersedia adalah: {df.columns.tolist()}")
        return None, None

    # df_train berisi baris dimana 'Risk Profile' TIDAK KOSONG
    df_train = df[df[target_column].notnull()].copy()

    # df_test berisi baris dimana 'Risk Profile' KOSONG
    df_test = df[df[target_column].isnull()].copy()

    print(f"Pemisahan berhasil:")
    print(f"  - Ukuran df_train: {df_train.shape}")
    print(f"  - Ukuran df_test : {df_test.shape}")

    return df_train, df_test

In [ ]:
df_train, df_test = split_by_risk_profile(df)

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
df_test = df_test.drop('Risk Profile', axis=1)
df_test.head()

## Handling Missing Values

In [ ]:
print("\nMissing df_train values per kolom:")
missing_values = df_train.isnull().sum()
missing_percentage = (missing_values / len(df_train)) * 100
missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percentage})
missing_data = missing_data[missing_data['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)
print(missing_data)

In [ ]:
df_train['Life Events'] = df_train['Life Events'].fillna('None')

In [ ]:
print("\nMissing df_train values per kolom:")
missing_values = df_train.isnull().sum()
missing_percentage = (missing_values / len(df_train)) * 100
missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percentage})
missing_data = missing_data[missing_data['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)
print(missing_data)

In [ ]:
categorical_cols_df_train = df_train.select_dtypes(include=['object']).columns.tolist()
imputer_most_frequent = SimpleImputer(strategy='most_frequent')
df_train[categorical_cols_df_train] = imputer_most_frequent.fit_transform(df_train[categorical_cols_df_train])
print("\nMissing df_train values after imputation (categorical columns):")
print(df_train[categorical_cols_df_train].isnull().sum())

In [ ]:
numerical_cols_df_train = df_train.select_dtypes(include=['float','int']).columns.tolist()
numerical_cols_df_train

In [ ]:
print("\nMissing df_test values per kolom:")
missing_values = df_test.isnull().sum()
missing_percentage = (missing_values / len(df_test)) * 100
missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percentage})
missing_data = missing_data[missing_data['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)
print(missing_data)

In [ ]:
df_test['Life Events'] = df_test['Life Events'].fillna('None')

In [ ]:
categorical_cols_df_test = df_test.select_dtypes(include=['object']).columns.tolist()
imputer_most_frequent = SimpleImputer(strategy='most_frequent')
df_test[categorical_cols_df_test] = imputer_most_frequent.fit_transform(df_test[categorical_cols_df_test])
print("\nMissing df_test values after imputation (categorical columns):")
print(df_test[categorical_cols_df_test].isnull().sum())

In [ ]:
numerical_cols_df_test = df_test.select_dtypes(include=['float','int']).columns.tolist()
numerical_cols_df_test

## Duplication

In [ ]:
print("Jumlah duplikasi di df_train:", df_train.duplicated().sum())

print("Jumlah duplikasi di df_test:", df_test.duplicated().sum())

## EDA

###  Analisis Variabel Target ('Risk Profile')

In [ ]:
# Assuming 'df_train' is your DataFrame and 'churn' is the column
risk_counts = df_train['Risk Profile'].value_counts()

# Create the pie chart
plt.figure(figsize=(8, 6))  # Adjust figure size if needed
plt.pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Risk Profile')
plt.legend(title="Risk", loc="best") # Added legend with title
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

### Histogram Distribusi Credit Score Pelanggan

In [ ]:
sns.histplot(df_train['Credit Score'], bins=30, kde=True)
plt.title('Distribusi Credit Score Pelanggan')
plt.xlabel('Credit Score)')
plt.ylabel('Nilai')
plt.show()

### Analisis Hubungan Fitur Numerik dengan 'Risk Profile'

In [ ]:
if df_train is not None:
    numerical_features = ['Age', 'Premium Amount', 'Coverage Amount', 'Credit Score']

    for feature in numerical_features:
        # Cek apakah fitur ada di DataFrame
        if feature in df_train.columns:
            plt.figure(figsize=(14, 6))

            # Plot 1: Histogram/KDE Plot untuk melihat distribusi keseluruhan
            plt.subplot(1, 2, 1)
            sns.histplot(df_train[feature], kde=True, bins=30)
            plt.title(f'Distribusi {feature}', fontsize=14)
            plt.xlabel(feature, fontsize=11)
            plt.ylabel('Frekuensi', fontsize=11)

            # Plot 2: Box Plot untuk melihat perbandingan antar Risk Profile
            plt.subplot(1, 2, 2)
            sns.boxplot(data=df_train, x='Risk Profile', y=feature, palette='pastel')
            plt.title(f'{feature} vs. Profil Risiko', fontsize=14)
            plt.xlabel('Profil Risiko', fontsize=11)
            plt.ylabel(feature, fontsize=11)

            plt.tight_layout() # Agar plot tidak tumpang tindih
            plt.show()
        else:
            print(f"Peringatan: Kolom '{feature}' tidak ditemukan.")
else:
    print("DataFrame EDA tidak tersedia, plot tidak dapat dibuat.")

### Analisis Hubungan Fitur Kategorikal dengan 'Risk Profile'

In [ ]:
if df_train is not None:
    categorical_features = ['Gender', 'Marital Status', 'Policy Type', 'Occupation']

    for feature in categorical_features:
         # Cek apakah fitur ada di DataFrame
        if feature in df_train.columns:
            plt.figure(figsize=(14, 6))

            # Plot 1: Count Plot untuk distribusi fitur itu sendiri
            plt.subplot(1, 2, 1)
            # Jika kategori terlalu banyak (misal: Occupation), tampilkan N teratas
            if df_train[feature].nunique() > 10:
                top_n = df_train[feature].value_counts().nlargest(10).index
                sns.countplot(data=df_train[df_train[feature].isin(top_n)], y=feature, order=top_n, palette='mako')
                plt.title(f'Distribusi {feature} (Top 10)', fontsize=14)
            else:
                sns.countplot(data=df_train, y=feature, order=df_train[feature].value_counts().index, palette='mako')
                plt.title(f'Distribusi {feature}', fontsize=14)
            plt.xlabel('Jumlah Nasabah', fontsize=11)
            plt.ylabel(feature, fontsize=11)

            # Plot 2: Count Plot dengan Hue 'Risk Profile' (proporsional)
            plt.subplot(1, 2, 2)
            # Jika kategori terlalu banyak, gunakan data yang sama (Top N)
            if df_train[feature].nunique() > 10:
                 sns.countplot(data=df_train[df_train[feature].isin(top_n)], y=feature, hue='Risk Profile', order=top_n, palette='viridis', dodge=True)
                 plt.title(f'{feature} vs. Profil Risiko (Top 10)', fontsize=14)
            else:
                 sns.countplot(data=df_train, y=feature, hue='Risk Profile', order=df_train[feature].value_counts().index, palette='viridis', dodge=True)
                 plt.title(f'{feature} vs. Profil Risiko', fontsize=14)
            plt.xlabel('Jumlah Nasabah', fontsize=11)
            plt.ylabel(feature, fontsize=11)
            plt.legend(title='Profil Risiko')


            plt.tight_layout()
            plt.show()
        else:
            print(f"Peringatan: Kolom '{feature}' tidak ditemukan.")
else:
    print("DataFrame EDA tidak tersedia, plot tidak dapat dibuat.")

### Heatmap Korelasi

In [ ]:
if df_train is not None:
    # Pilih hanya kolom numerik (hati-hati, ini mungkin perlu disesuaikan)
    numerical_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

    if not numerical_cols.empty:
        plt.figure(figsize=(10, 8))
        correlation_matrix = df_train[numerical_cols].corr()
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
        plt.title('Heatmap Korelasi Fitur Numerik', fontsize=16)
        plt.show()
    else:
        print("Tidak ada kolom numerik yang ditemukan untuk heatmap korelasi.")
else:
    print("DataFrame EDA tidak tersedia, plot tidak dapat dibuat.")

## Feature Engineering

In [ ]:
df_train['Policy_Duration'] = (df_train['Policy Renewal Date'] - df_train['Policy Start Date']).dt.days
df_test['Policy_Duration'] = (df_test['Policy Renewal Date'] - df_test['Policy Start Date']).dt.days

df_train['Start_Year'] = df_train['Policy Start Date'].dt.year
df_test['Start_Year'] = df_test['Policy Start Date'].dt.year

df_train['Start_Month'] = df_train['Policy Start Date'].dt.month
df_test['Start_Month'] = df_test['Policy Start Date'].dt.month

In [ ]:
df_train['Premium_Coverage_Ratio'] = df_train['Premium Amount'] / (df_train['Coverage Amount'])
df_test['Premium_Coverage_Ratio'] = df_test['Premium Amount'] / (df_test['Coverage Amount'])

df_train['Deductible_Coverage_Ratio'] = df_train['Deductible'] / (df_train['Coverage Amount'] )
df_test['Deductible_Coverage_Ratio'] = df_test['Deductible'] / (df_test['Coverage Amount'] )

df_train['Premium_Age_Ratio'] = df_train['Premium Amount'] / (df_train['Age'])
df_test['Premium_Age_Ratio'] = df_test['Premium Amount'] / (df_test['Age'])

df_train['Age_Credit_Interaction'] = df_train['Age'] * df_train['Credit Score']
df_test['Age_Credit_Interaction'] = df_test['Age'] * df_test['Credit Score']

### Drop Kolom yang date

In [ ]:
date_columns = df_train.select_dtypes(include=['datetime64[ns]']).columns

df_train = df_train.drop(columns=date_columns)
df_test = df_test.drop(columns=date_columns)

### Normalisasi

In [ ]:
scaler_test = MinMaxScaler()
df_test[numerical_cols_df_test] = scaler_test.fit_transform(df_test[numerical_cols_df_test])

print("\nDataFrame df_test after normalization:")
print(df_test.head())

scaler_train = MinMaxScaler()

scaler_train.fit(df_train[numerical_cols_df_train])

df_train[numerical_cols_df_train] = scaler_train.transform(df_train[numerical_cols_df_train])
df_test[numerical_cols_df_test] = scaler_train.transform(df_test[numerical_cols_df_test]) # Use the same scaler fitted on train

df_train.head()

In [ ]:
df_test.head()

### Encoding Variabel Kategorik

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Tandai data train dan test
df_train['is_train'] = 1
df_test['is_train'] = 0

# Gabungkan
combined = pd.concat([df_train, df_test])

# Mapping manual untuk risk_profile
risk_profile_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
combined['Risk Profile'] = combined['Risk Profile'].map(risk_profile_mapping)

# Deteksi semua kolom object KECUALI 'risk_profile'
object_cols = combined.select_dtypes(include='object').columns.drop('Risk Profile', errors='ignore')

# Encode kolom selain risk_profile
le = LabelEncoder()
for col in object_cols:
    combined[col] = le.fit_transform(combined[col].astype(str))

# Pisahkan kembali
df_train_encoded = combined[combined['is_train'] == 1].drop(columns=['is_train'])
df_test_encoded = combined[combined['is_train'] == 0].drop(columns=['is_train'])

In [ ]:
df_train_encoded.head()

In [ ]:
df_test_encoded.head()

In [ ]:
df_train_encoded = df_train_encoded.drop('Customer ID', axis=1)
df_test_encoded = df_test_encoded.drop('Customer ID', axis=1)

df_train_encoded.head()

In [ ]:
df_test_encoded.head()

## Modelling

### Resampling and Splitting

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
X = df_train_encoded.drop('Risk Profile', axis=1)
y = df_train_encoded['Risk Profile']

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split data train_encoded menjadi training dan validation set
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_val:", X_val.shape)
print("Ukuran y_train:", y_train.shape)
print("Ukuran y_val:", y_val.shape)

### Random Forest

In [ ]:
# Inisialisasi model Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, # Jumlah pohon dalam forest
                                  random_state=42, # Untuk reproducibility
                                  class_weight='balanced' # Menangani imbalansi kelas jika ada
                                 )

# Latih model menggunakan data training
rf_model.fit(X_train, y_train)

# Prediksi pada data validasi
y_pred = rf_model.predict(X_val)

# Evaluasi model
print("Classification Report:")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

dt_model = DecisionTreeClassifier(random_state=42) # Untuk reproducibility

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_val)

# Evaluasi model Decision Tree
print("Classification Report (Decision Tree):")
print(classification_report(y_val, y_pred_dt))

print("\nConfusion Matrix (Decision Tree):")
print(confusion_matrix(y_val, y_pred_dt))

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize and train a Logistic Regression model
logreg_model = LogisticRegression(random_state=42, max_iter=1000) # You can adjust hyperparameters here
logreg_model.fit(X_train, y_train)

# Make predictions on the test set
logreg_y_pred = logreg_model.predict(X_val)

# Evaluate the model
print(classification_report(y_val, logreg_y_pred))
print(confusion_matrix(y_val, logreg_y_pred))



### XGBoost

In [ ]:
import xgboost as xgb

# Initialize and train an XGBoost classifier
xgb_model = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss') # You can adjust hyperparameters here
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
xgb_y_pred = xgb_model.predict(X_val)

# Evaluate the model
print(classification_report(y_val, xgb_y_pred))
print(confusion_matrix(y_val, xgb_y_pred))


### LightGBM

In [ ]:
!pip install lightgbm

In [ ]:
import lightgbm as lgb

# Initialize and train a LightGBM classifier
lgb_model = lgb.LGBMClassifier(random_state=42)  # You can adjust hyperparameters here
lgb_model.fit(X_train, y_train)

# Make predictions on the test set
lgb_y_pred = lgb_model.predict(X_val)

# Evaluate the model
print(classification_report(y_val, lgb_y_pred))
print(confusion_matrix(y_val, lgb_y_pred))

### CatBoost

In [ ]:
!pip install catboost

from catboost import CatBoostClassifier

In [ ]:
# Initialize and train a CatBoost classifier
catboost_model = CatBoostClassifier(random_state=42, verbose=0) # verbose=0 hides training output
catboost_model.fit(X_train, y_train)

# Make predictions on the test set
catboost_y_pred = catboost_model.predict(X_val)

# Evaluate the model
print("Classification Report (CatBoost):")
print(classification_report(y_val, catboost_y_pred))

print("\nConfusion Matrix (CatBoost):")
print(confusion_matrix(y_val, catboost_y_pred))

## Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold


kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(rf_model, X_train, y_train, cv=kfold, scoring='accuracy')

# 4. Tampilkan Hasil
print("Hasil Cross-Validation (5-Fold):")
print(f"Skor Akurasi untuk setiap fold: {scores}")
print(f"Rata-rata Akurasi: {scores.mean():.4f}")
print(f"Standar Deviasi Akurasi: {scores.std():.4f}")

In [ ]:
from sklearn.model_selection import cross_val_predict

print("\n--- Cross-Validation Prediction Report on Training Data ---")
y_pred_cv = cross_val_predict(rf_model, X_train, y_train, cv=kfold)

# Classification report berdasarkan prediksi cross-validation
print("Classification Report (Cross-Validated):")
print(classification_report(y_train, y_pred_cv))

# Confusion matrix dari hasil cross-validation
print("\nConfusion Matrix (Cross-Validated):")
print(confusion_matrix(y_train, y_pred_cv))

In [ ]:
sns.heatmap(confusion_matrix(y_train, y_pred_cv), annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Cross-Validation)")
plt.show()

## Hyperparameter Tuning

### Optuna

In [ ]:
!pip install optuna

import optuna

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),  # lebih sempit
        'max_depth': trial.suggest_int('max_depth', 5, 20),          # lebih sempit
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
    }

    model = RandomForestClassifier(random_state=42, class_weight='balanced', **params)
    # Ganti ke cv=3 untuk lebih cepat
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1).mean()
    return score

In [ ]:
# Proses optimasi
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# Tampilkan hasil terbaik
best_params = study.best_trial.params
print(f"\nBest CV Accuracy: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, val in best_params.items():
    print(f"  {key}: {val}")

In [ ]:
# Latih model dengan parameter terbaik
best_rf = RandomForestClassifier(random_state=42, class_weight='balanced', **best_params)
best_rf.fit(X_train, y_train)
y_pred = best_rf.predict(X_val)

# Evaluasi pada validation set
print("\n--- Validation Performance ---")
print(classification_report(y_val, y_pred))
cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()